In [15]:
from newsapi import NewsApiClient
import yfinance as yf

from Utils.clean_data_helper import *

'''
If unable to install pygooglenews, try
- pip install "setuptools<58.0"
- pip install feedparser --upgrade
- pip install setuptools
- pip install pygooglenews --upgrade
'''
from pygooglenews import GoogleNews
from tqdm import tqdm
import requests
import json

import warnings
warnings.filterwarnings('ignore')


# Get S&P500 companies' tickers

In [4]:
# response = requests.get('https://stockanalysis.com/list/sp-500-stocks/')
# companies_info_df = pd.read_html(response.content)[0]
# companies_info_df = companies_info_df.drop(columns=["No.", "Stock Price", "% Change", "Revenue"])
# companies_info_df.to_csv("Data Updated/SPY_companies_info.csv", index=False)

In [5]:
# # tech stocks, pharma stocks, oil stocks, tobacco stocks and Market
# tickers = companies_info_df["Symbol"].to_list()

# News Data Download
- newsapi has a limit of 2024-11-01 onwards
- pygoognews https://github.com/kotartemiy/pygooglenews
- Some other packages to try out: https://www.newscatcherapi.com/blog/python-web-scraping-libraries-to-mine-news-data

In [8]:
# # Looping over 500 tickers: this run will take a few hours
# gn = GoogleNews(lang = 'en')
#
# news_dfs = []
# for ticker in tqdm(tickers):
#     top = gn.search(ticker)
#     entries = top["entries"]
#     df_temp = clean_goog_news(entries)
#     df_temp["date"] = df_temp["date"].apply(lambda d: pd.to_datetime(d, errors='coerce').date())
#     df_temp = df_temp.dropna()
#     df_temp["ticker"] = ticker
#     news_dfs += [df_temp.copy()]
#
# goog_news_df = pd.concat(news_dfs, ignore_index=True)
# goog_news_df

100%|██████████| 503/503 [3:41:23<00:00, 26.41s/it]  


,date,title,source,ticker
0,2024-12-10,Analyst Says Apple (AAPL) Stock is Not a ‘Holi...,https://news.google.com/rss/articles/CBMigAFBV...,AAPL
1,2024-12-11,S&P 500 Gains and Losses Today: Broadcom Soars...,https://news.google.com/rss/articles/CBMivgFBV...,AAPL
2,2024-12-11,Apple's AI Breakthrough: Broadcom Shares Soar ...,https://news.google.com/rss/articles/CBMijAFBV...,AAPL
3,2024-12-12,Apple Moves Ahead In AI With New Broadcom Deal...,https://news.google.com/rss/articles/CBMikAFBV...,AAPL
4,2024-12-12,Apple (AAPL) Is Developing Its Own Custom Micr...,https://news.google.com/rss/articles/CBMirgFBV...,AAPL
...,...,...,...,...
49661,2018-01-24,"""America's Next Top Model"" Makes Contestant Wi...",https://news.google.com/rss/articles/CBMilgFBV...,AMTM
49662,2022-04-12,'America's Next Top Model' Alums Allege Trauma...,https://news.google.com/rss/articles/CBMi5AFBV...,AMTM
49663,2021-12-30,Why ‘America’s Next Top Model’ is getting slam...,https://news.google.com/rss/articles/CBMijwFBV...,AMTM
49664,2021-12-05,America's Next Top Model: Tyra Banks' 8 Iconic...,https://news.google.com/rss/articles/CBMifEFVX...,AMTM


In [27]:
# goog_news_df.to_csv("Data Updated/goog_news_df.csv")

In [28]:
# # Clean and format dataset
# def remove_ticker_substrings(text):
#     for ticker in tickers:
#         text = text.replace(f"({ticker})", "")
#     return text
#
# goog_news_df = goog_news_df.dropna()
# goog_news_df = goog_news_df.drop_duplicates().reset_index(drop=True)
# goog_news_df["date"] = goog_news_df["date"].apply(lambda d: pd.to_datetime(d, errors='coerce', utc=True).date())
# goog_news_df["title"] = goog_news_df["title"].str.replace(r"\('\w+':'\w+'\)", "", regex=True)
# goog_news_df["title"] = remove_ticker_substrings(goog_news_df["title"])

# CapIQ Data
Financial data is acquired from Capital IQ from dates 01-01-2024 to 04-12-2024

In [16]:
capiq_df = pd.read_excel(r"C:\Users\Tylus\Desktop\capiq_news_data.xlsx")
capiq_df = capiq_df.rename(columns = {"Key Developments By Date": "date",
                            "Key Development Headline": "title",
                            "Key Development Sources": "source",
                            "Primary Industry": "topic"
                            })
capiq_df =  capiq_df.drop(columns=["Key Developments by Type", "Key Development Situation"])

In [17]:
# Keep only headlines for individual stocks and get ticker
capiq_df["ticker"] = capiq_df["Company Name(s)"].str.findall(r'\([A-Za-z]+:[A-Za-z]+\)').apply(','.join)
capiq_df = capiq_df[capiq_df["ticker"].str.count(',') == 0]
capiq_df["ticker"] = capiq_df["ticker"].str.extract(r':([A-Za-z]+)\)')

# Clean and format dataset
capiq_df = capiq_df[["date", "title", "source", "ticker"]]
capiq_df = capiq_df.dropna()
capiq_df = capiq_df.drop_duplicates().reset_index(drop=True)
capiq_df["title"] = capiq_df["title"].str.replace(r"\([A-Za-z]+:[A-Za-z]+\)", "", regex=True)
capiq_df

,date,title,source,ticker
0,2014-01-01,Motorola Solutions to Provide IDF's Battlefiel...,Other,MSI
1,2014-01-01,TE Connectivity Brings Advanced Mobile Service...,Business Wire,TEL
2,2014-01-01,Leidos Holdings Receives Follow-On Contract fr...,Datamonitor NewsWire,LDOS
3,2014-01-01,MarketAxess Holdings Inc.'s Equity Buyback ann...,Capital IQ Buybacks Database,MKTX
4,2014-01-02,Southwest Airlines Orders Aviation Partners Bo...,PR Newswire,LUV
...,...,...,...,...
408745,2024-12-04,CSX Corporation Presents at UBS Global Industr...,PR Newswire; Business Wire; GlobeNewswire; Com...,CSX
408746,2024-12-04,UnitedHealth Group Incorporated - Analyst/Inve...,Business Wire,UNH
408747,2024-12-04,LyondellBasell Industries N.V. Presents at Gol...,PR Newswire; Business Wire; GlobeNewswire; Com...,LYB
408748,2024-12-04,Freeport-McMoRan Inc. Presents at Mines and Mo...,Company Website,FCX


In [18]:
capiq_df.to_csv("Data Updated/capiq_df.csv")

# Merging CapIQ and PyGoogleNews

In [ ]:
# news_all_df = goog_news_df.append(capiq_df).sort_values("date", ascending=True).reset_index(drop=True)
# news_all_df["date"] = news_all_df["date"].apply(lambda d: pd.to_datetime(d, errors='coerce'))
# news_all_df

In [72]:
# news_all_df.to_csv("Data Updated/news_all_df.csv")

# Train Test Split and Removing Similar News

In [19]:
news_all_df = capiq_df
news_all_df["date"] = news_all_df["date"].apply(lambda d: pd.to_datetime(d, errors='coerce'))

In [20]:
# Split dataset in train set (for headlines before 2024) and test set (for headlines in 2024)
news_train_df = news_all_df[news_all_df["date"].dt.year < 2024].copy()
news_test_df = news_all_df[news_all_df["date"].dt.year >= 2024].copy()

In [ ]:
news_all_df.to_csv("Data Updated/news_all_df.csv")
news_train_df.to_csv("Data Updated/news_train_df.csv")
news_test_df.to_csv("Data Updated/_news_test_df.csv")

In [25]:
def remove_similar_news_optimised(df, col, date_col, threshold=0.80, window_days=20):
    df = df.sort_values(by=date_col).reset_index(drop=True)

    vectorizer = CountVectorizer()
    vectors = vectorizer.fit_transform(df[col])
    cos_sim = cosine_similarity(vectors)

    dates = pd.to_datetime(df[date_col])
    to_remove = set()

    for r in range(len(cos_sim)):
        if r in to_remove:
            continue

        for c in range(r + 1, len(cos_sim)):
            if c in to_remove:
                continue

            # Check if the two entries are within the date window
            if abs((dates.iloc[r] - dates.iloc[c]).days) > window_days:
                continue

            # Check similarity
            if cos_sim[r, c] > threshold:
                to_remove.add(c)

    valid_indices = set(df.index)
    to_remove = to_remove.intersection(valid_indices)
    df = df.drop(index=to_remove).reset_index(drop=True)
    return df

In [26]:
dfs = [news_train_df, news_test_df]

for idx, df in enumerate(dfs, start=1):
    chunked_results = []
    total_tickers = len(df['ticker'].unique())

    for i, (ticker, group) in enumerate(df.groupby("ticker"), start=1):
        print(f"Processing ticker {i}/{total_tickers}: {ticker} with {len(group)} articles...")
        group = group.sort_values(by="date").reset_index(drop=True)
        filtered_group = remove_similar_news_optimised(group, "title", "date", threshold=0.80, window_days=20)
        chunked_results.append(filtered_group)
        print(f"Completed processing ticker {i}/{total_tickers}: {ticker}. Filtered articles: {len(filtered_group)}.")

    filtered_df = pd.concat(chunked_results).reset_index(drop=True)
    print(f"Finished processing all groups in dataset {idx}. Final DataFrame has {len(filtered_df)} articles.")

    if idx == 1:
        filtered_news_train_df = filtered_df
    elif idx == 2:
        filtered_news_test_df = filtered_df

Processing ticker 1/516: A with 951 articles...
Completed processing ticker 1/516: A. Filtered articles: 924.
Processing ticker 2/516: AAL with 1 articles...
Completed processing ticker 2/516: AAL. Filtered articles: 1.
Processing ticker 3/516: AAPL with 1557 articles...
Completed processing ticker 3/516: AAPL. Filtered articles: 1339.
Processing ticker 4/516: ABBV with 1522 articles...
Completed processing ticker 4/516: ABBV. Filtered articles: 1438.
Processing ticker 5/516: ABI with 1 articles...
Completed processing ticker 5/516: ABI. Filtered articles: 1.
Processing ticker 6/516: ABNB with 314 articles...
Completed processing ticker 6/516: ABNB. Filtered articles: 292.
Processing ticker 7/516: ABT with 876 articles...
Completed processing ticker 7/516: ABT. Filtered articles: 840.
Processing ticker 8/516: ACGL with 459 articles...
Completed processing ticker 8/516: ACGL. Filtered articles: 423.
Processing ticker 9/516: ACN with 1913 articles...
Completed processing ticker 9/516: AC

In [29]:
filtered_news_train_df

,date,title,source,ticker
0,2014-01-06,Agilent Technologies Introduces New Version of...,Business Wire,A
1,2014-01-06,Agilent Technologies Introduces ICP-MS and MP-...,Business Wire,A
2,2014-01-08,Agilent Technologies Inc. Introduces New Exter...,Business Wire,A
3,2014-01-09,AT4 Wireless Selects Agilent Technologies Test...,Business Wire,A
4,2014-01-09,Agilent Technologies Introduces First USB 3.0 ...,Other,A
...,...,...,...,...
332971,2023-11-02,"Zoetis Inc., Q3 2023 Earnings Call, Nov 02, 2023",Business Wire,ZTS
332972,2023-11-02,Zoetis Inc. Provides Earnings Guidance for the...,Business Wire,ZTS
332973,2023-11-02,Zoetis Inc. Reports Earnings Results for the T...,S&P Capital IQ Financials Database,ZTS
332974,2023-11-28,Zoetis Inc. Presents at Piper Sandler 35th Ann...,PR Newswire; Business Wire; Other; GlobeNewswi...,ZTS


In [30]:
filtered_news_test_df

,date,title,source,ticker
0,2024-01-08,"Agilent Technologies, Inc. Presents at BIO Par...",Business Wire; GlobeNewswire; Company Website,A
1,2024-01-09,"Agilent Technologies, Inc. Presents at J.P. Mo...",PR Newswire; Business Wire; Other; GlobeNewswi...,A
2,2024-01-16,"Agilent Technologies, Inc. Announces New Prote...",Business Wire,A
3,2024-01-16,"Agilent Technologies, Inc. Presents at 23rd An...",Company Website,A
4,2024-01-17,"Agilent Technologies, Inc. - Special Call",Company Website,A
...,...,...,...,...
29276,2024-11-04,Zoetis Inc. Raises Revenue Guidance for the Fu...,Business Wire,ZTS
29277,2024-11-11,Zoetis Inc. Presents at Fortune Global Forum 2...,PR Newswire,ZTS
29278,2024-11-11,Zoetis Inc. Announces Executive Changes,Business Wire,ZTS
29279,2024-11-20,Zoetis Inc. Presents at Jefferies London Healt...,PR Newswire; Business Wire; Other; GlobeNewswi...,ZTS


In [31]:
filtered_news_train_df.to_csv("Data Updated/filtered_news_train_df.csv")
filtered_news_test_df.to_csv("Data Updated/filtered_news_test_df.csv")

# Stocks Data Download
- Note that this is not the returns of the stocks data
- Let's focus on periods 2023-01-01 to 2024-12-01 for this project


In [99]:
stocks = yf.download(tickers, threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BRK.B']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
['BF.B']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (1d 2014-01-01 -> 2024-12-01)')


In [100]:
stocks

Ticker             JNJ                                                  \
Price             Open        High         Low       Close   Adj Close   
Date                                                                     
2014-01-02   91.139999   91.730003   91.010002   91.029999   67.211365   
2014-01-03   91.290001   92.220001   91.260002   91.849998   67.816818   
2014-01-06   91.930000   92.750000   91.870003   92.330002   68.171196   
2014-01-07   93.040001   94.620003   92.989998   94.290001   69.618393   
2014-01-08   94.169998   94.599998   93.879997   94.160004   69.522385   
...                ...         ...         ...         ...         ...   
2024-11-22  155.899994  157.119995  154.110001  155.169998  153.934845   
2024-11-25  155.169998  157.039993  155.139999  155.779999  154.539993   
2024-11-26  155.160004  155.250000  153.160004  154.520004  154.520004   
2024-11-27  154.630005  156.630005  154.600006  155.399994  155.399994   
2024-11-29  154.889999  155.669998  154.169998  155.009995  155.009995   

Ticker                       FRT                                      ...  \
Price         Volume        Open        High         Low       Close  ...   
Date                                                                  ...   
2014-01-02   5919600  101.379997  101.839996  100.370003  100.900002  ...   
2014-01-03   5637600  101.129997  102.279999  100.970001  101.930000  ...   
2014-01-06   7443500  103.050003  103.580002  102.269997  103.260002  ...   
2014-01-07  11063300  103.339996  104.360001  101.800003  103.970001  ...   
2014-01-08   9096700  103.709999  103.919998  102.070000  103.470001  ...   
...              ...         ...         ...         ...         ...  ...   
2024-11-22   8266000  114.970001  115.199997  114.430000  114.860001  ...   
2024-11-25  12256200  115.669998  116.580002  115.330002  115.980003  ...   
2024-11-26   5683500  116.070000  116.209999  115.150002  115.900002  ...   
2024-11-27   4140400  116.629997  118.000000  116.419998  117.529999  ...   
2024-11-29   5687800  117.599998  118.089996  116.610001  116.650002  ...   

Ticker             EFX                                          HUM  \
Price              Low       Close   Adj Close   Volume        Open   
Date                                                                  
2014-01-02   68.199997   68.449997   61.218979   336300  102.760002   
2014-01-03   68.550003   68.900002   61.621445   314800  102.860001   
2014-01-06   68.190002   68.260002   61.049038   284800  102.300003   
2014-01-07   68.320000   68.730003   61.469414   320900   98.720001   
2014-01-08   67.910004   68.739998   61.478336   445800  100.660004   
...                ...         ...         ...      ...         ...   
2024-11-22  252.660004  253.580002  253.580002   773100  296.000000   
2024-11-25  256.170013  263.890015  263.890015  1732000  312.000000   
2024-11-26  256.829987  258.940002  258.940002  1113900  304.000000   
2024-11-27  260.529999  261.190002  261.190002   559900  296.670013   
2024-11-29  261.390015  261.559998  261.559998   734600  294.920013   

Ticker                                                               
Price             High         Low       Close   Adj Close   Volume  
Date                                                                 
2014-01-02  103.879997  102.300003  102.839996   95.121132  1393500  
2014-01-03  103.040001  101.580002  101.760002   94.122238  1234400  
2014-01-06  102.379997   99.959999  100.760002   93.197266  1850200  
2014-01-07  101.349998   98.500000  100.550003   93.003052  2561700  
2014-01-08  100.660004   98.949997   99.400002   91.939339  2332700  
...                ...         ...         ...         ...      ...  
2024-11-22  303.029999  295.410004  298.109985  298.109985  1450600  
2024-11-25  313.000000  303.630005  304.179993  304.179993  2656600  
2024-11-26  304.000000  294.470001  295.589996  295.589996  1556000  
2024-11-27  299.190002  294.959991  296.679993  2

In [ ]:
# ticker_tz_df = pd.DataFrame()
# for t in tickers:
#     try:
#         ticker_info = yf.Ticker(t)
#         ticker_tz = ticker_info.info['timeZoneShortName']
#         ticker_tz_temp_df = pd.DataFrame({'Ticker':[t], 'tz':[ticker_tz]})
#         ticker_tz_df = pd.concat([ticker_tz_df, ticker_tz_temp_df])
#     except:
#         print(f'No timezone info for {t}')

No timezone info for BRK.B


In [ ]:
# print(f'Timezones of SPY stocks include: {ticker_tz_df.tz.unique()}')

Timezones of SPY stocks include: ['EST']


In [101]:
# Convert Stocks timezone to UTC 0
stocks.index = (
    stocks.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [102]:
market = yf.download(["SPY"], threads=True, group_by='ticker', start='2014-01-01', end='2024-12-01', multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [103]:
market

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-01-02,183.979996,184.070007,182.479996,182.919998,151.242920,119636900
2014-01-03,183.229996,183.600006,182.630005,182.889999,151.218094,81390600
2014-01-06,183.490005,183.559998,182.080002,182.360001,150.779907,108028200
2014-01-07,183.089996,183.789993,182.949997,183.479996,151.705978,86144200
2014-01-08,183.449997,183.830002,182.889999,183.520004,151.739014,96582300
...,...,...,...,...,...,...
2024-11-22,593.659973,596.150024,593.150024,595.510010,595.510010,38226400
2024-11-25,599.520020,600.859985,595.200012,597.530029,597.530029,42441400
2024-11-26,598.799988,601.330017,598.070007,600.650024,600.650024,45621300


In [104]:
market.index = (
    market.index
    .tz_localize("EST")
    .tz_convert("UTC")
    .tz_localize(None)
    .normalize()
)

In [105]:
stocks

Ticker             JNJ                                                  \
Price             Open        High         Low       Close   Adj Close   
Date                                                                     
2014-01-02   91.139999   91.730003   91.010002   91.029999   67.211365   
2014-01-03   91.290001   92.220001   91.260002   91.849998   67.816818   
2014-01-06   91.930000   92.750000   91.870003   92.330002   68.171196   
2014-01-07   93.040001   94.620003   92.989998   94.290001   69.618393   
2014-01-08   94.169998   94.599998   93.879997   94.160004   69.522385   
...                ...         ...         ...         ...         ...   
2024-11-22  155.899994  157.119995  154.110001  155.169998  153.934845   
2024-11-25  155.169998  157.039993  155.139999  155.779999  154.539993   
2024-11-26  155.160004  155.250000  153.160004  154.520004  154.520004   
2024-11-27  154.630005  156.630005  154.600006  155.399994  155.399994   
2024-11-29  154.889999  155.669998  154.169998  155.009995  155.009995   

Ticker                       FRT                                      ...  \
Price         Volume        Open        High         Low       Close  ...   
Date                                                                  ...   
2014-01-02   5919600  101.379997  101.839996  100.370003  100.900002  ...   
2014-01-03   5637600  101.129997  102.279999  100.970001  101.930000  ...   
2014-01-06   7443500  103.050003  103.580002  102.269997  103.260002  ...   
2014-01-07  11063300  103.339996  104.360001  101.800003  103.970001  ...   
2014-01-08   9096700  103.709999  103.919998  102.070000  103.470001  ...   
...              ...         ...         ...         ...         ...  ...   
2024-11-22   8266000  114.970001  115.199997  114.430000  114.860001  ...   
2024-11-25  12256200  115.669998  116.580002  115.330002  115.980003  ...   
2024-11-26   5683500  116.070000  116.209999  115.150002  115.900002  ...   
2024-11-27   4140400  116.629997  118.000000  116.419998  117.529999  ...   
2024-11-29   5687800  117.599998  118.089996  116.610001  116.650002  ...   

Ticker             EFX                                          HUM  \
Price              Low       Close   Adj Close   Volume        Open   
Date                                                                  
2014-01-02   68.199997   68.449997   61.218979   336300  102.760002   
2014-01-03   68.550003   68.900002   61.621445   314800  102.860001   
2014-01-06   68.190002   68.260002   61.049038   284800  102.300003   
2014-01-07   68.320000   68.730003   61.469414   320900   98.720001   
2014-01-08   67.910004   68.739998   61.478336   445800  100.660004   
...                ...         ...         ...      ...         ...   
2024-11-22  252.660004  253.580002  253.580002   773100  296.000000   
2024-11-25  256.170013  263.890015  263.890015  1732000  312.000000   
2024-11-26  256.829987  258.940002  258.940002  1113900  304.000000   
2024-11-27  260.529999  261.190002  261.190002   559900  296.670013   
2024-11-29  261.390015  261.559998  261.559998   734600  294.920013   

Ticker                                                               
Price             High         Low       Close   Adj Close   Volume  
Date                                                                 
2014-01-02  103.879997  102.300003  102.839996   95.121132  1393500  
2014-01-03  103.040001  101.580002  101.760002   94.122238  1234400  
2014-01-06  102.379997   99.959999  100.760002   93.197266  1850200  
2014-01-07  101.349998   98.500000  100.550003   93.003052  2561700  
2014-01-08  100.660004   98.949997   99.400002   91.939339  2332700  
...                ...         ...         ...         ...      ...  
2024-11-22  303.029999  295.410004  298.109985  298.109985  1450600  
2024-11-25  313.000000  303.630005  304.179993  304.179993  2656600  
2024-11-26  304.000000  294.470001  295.589996  295.589996  1556000  
2024-11-27  299.190002  294.959991  296.679993  2

In [106]:
stocks.to_csv("Data Updated/stocks_data.csv")

In [107]:
market

,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,
2014-01-02,183.979996,184.070007,182.479996,182.919998,151.242920,119636900
2014-01-03,183.229996,183.600006,182.630005,182.889999,151.218094,81390600
2014-01-06,183.490005,183.559998,182.080002,182.360001,150.779907,108028200
2014-01-07,183.089996,183.789993,182.949997,183.479996,151.705978,86144200
2014-01-08,183.449997,183.830002,182.889999,183.520004,151.739014,96582300
...,...,...,...,...,...,...
2024-11-22,593.659973,596.150024,593.150024,595.510010,595.510010,38226400
2024-11-25,599.520020,600.859985,595.200012,597.530029,597.530029,42441400
2024-11-26,598.799988,601.330017,598.070007,600.650024,600.650024,45621300


In [108]:
market.to_csv("Data Updated/market_data.csv")